In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.6 Perplexity and evaluation

> 🎯 **Goal:** Translate raw loss into perplexity, a human-readable score, and learn to judge a model against the right baseline instead of staring at a bare number.

**Why this matters.** Loss is the number the optimizer minimizes, but as a report card it is hard to interpret: is a loss of 2.3 good? You cannot tell without context. Perplexity turns that abstract loss into something you can reason about and compare across models. Every serious evaluation of a language model reports perplexity, so this is the metric you will actually quote when you say how good your PocketLM is.

**The intuition.** Perplexity answers a concrete question: when the model predicts the next character, how many equally-likely options is it effectively choosing among? A perplexity of 5 means the model is about as uncertain as someone guessing uniformly between 5 choices. Lower is better, the model has narrowed the field. A model that has learned nothing and spreads its bet evenly over all `V` possible characters has perplexity exactly `V`. So `V` is the do-nothing baseline, and the whole game is to score well below it.

**The mechanics.** Perplexity is simply `exp(loss)`, the exponential of the loss measured in nats (the natural-log information unit from lesson 1). The exponential undoes the log, converting "average surprise in nats" back into an effective count of choices. Because `ln(V)` is the loss of uniform guessing, `exp(ln(V)) = V` is its perplexity, which is why the uniform baseline equals the vocabulary size. We train the model briefly, measure its loss on the data with `estimate_loss` (averaged over several batches for stability), and convert it with `perplexity(loss)`. The number to beat is `tok.vocab_size`.

In [ ]:
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss, perplexity)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                     n_head=2, n_embd=64)
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
steps = 150 if os.environ.get("POCKETLM_CI") else 400
for _ in range(steps):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
loss = estimate_loss(model, data, 32, 16, iters=20, generator=g)
print("loss:", round(loss, 3), " perplexity:", round(perplexity(loss), 2))
print("uniform baseline perplexity:", tok.vocab_size)

**What just happened.** The printout shows the measured loss, its perplexity, and the uniform baseline (the vocabulary size). Even this tiny model, trained for only a few hundred steps, lands a perplexity well under the baseline. That gap is the proof: the model has discovered real structure in the text (which characters tend to follow which) rather than guessing blindly. It has effectively shrunk its menu of plausible next characters.

⚠️ **Common pitfalls**
- Quoting a perplexity with no baseline: "perplexity 18" is meaningless until you say the vocabulary was 65 and the baseline was 65.
- Comparing perplexities across different tokenizers or vocabularies: the baseline differs, so the numbers are not comparable. Only compare like with like.
- Measuring perplexity on the training set and calling it evaluation: report it on held-out data for an honest reading (here we measure on the full `data` for simplicity, but in practice use the validation split).

🏋️ **Try it yourself.** Increase `steps` (try 800 locally) and watch the perplexity drop further toward 1, the model getting more confident and more correct. Then compute perplexity on a fresh untrained model (skip the training loop) and confirm it sits right at the baseline, exactly the do-nothing score.

In [ ]:
assert perplexity(loss) < tok.vocab_size